# Deteksi dan Pengenalan Pelat Nomor Kendaraan — Tahap 2: evaluasi dan perbandingan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-uji-t1/blob/main/notebooks/04_evaluasi.ipynb)

Notebook ini menyusun tabel untuk README bagian *Hasil*, menjalankan satu uji ablasi, dan menampilkan kasus gagal. Jalankan setelah notebook 02 dan 03 menyimpan hasilnya.

In [1]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-uji-t1"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-uji-t1")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas', 'x', 'y', 'w', 'h', 'teks_pelat']
JUMLAH_ANGGOTA = 3
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Citra   : /tmp/tmpezs920av/repo/data/sample
Anotasi : /tmp/tmpezs920av/repo/data/anotasi.csv (ada)


In [2]:
anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
UJI_CEPAT = None  # isi angka (mis. 20) saat mencoba-coba agar cepat; kembalikan ke None untuk hasil akhir
print(len(anotasi), "baris anotasi")

6 baris anotasi


## 1. Tabel perbandingan untuk README

Salin tabel yang tercetak ke README bagian **Hasil (Tahap 2)**.

In [3]:
semua = ev.baca_hasil(BERKAS_HASIL)
if not semua:
    raise RuntimeError("Belum ada hasil. Jalankan notebook 02 dan 03 dengan UJI_CEPAT = None.")
print(ev.tabel_markdown(semua, ['IoU', 'CER']))

| Metode | Jenis | IoU | CER |
| --- | --- | --- | --- |
| Dummy klasik 1 | klasik | 0.000 | 1.000 |
| Dummy klasik 2 | klasik | 0.000 | 1.000 |
| Dummy modern | modern | 0.000 | 1.000 |


## 2. Uji ablasi

Ubah satu hal dan ukur pengaruhnya, misalnya **ukuran kernel morfologi** dengan nilai [3, 5, 7, 9]. Salin fungsi metode terbaik dari notebook 02/03 ke sel di bawah dan beri parameter.

In [4]:
def metode_terbaik(citra_bgr, parameter):
    """Metode terbaik dengan satu parameter yang diuji.

    Masukan: citra BGR uint8. Keluaran: (kotak, teks) — kotak = (x, y, w, h) pelat atau None; teks = karakter pelat.
    """
    return None, ""  # DUMMY: tidak mendeteksi apa pun; ganti dengan metode kelompok

In [5]:
NILAI_PARAMETER = [3, 5, 7, 9]
ablasi = {}
for p in NILAI_PARAMETER:
    hasil = ev.evaluasi_pelat(lambda c: metode_terbaik(c, p), anotasi, FOLDER_CITRA, batas=UJI_CEPAT)
    ablasi[p] = hasil['metrik']
    print(p, hasil['metrik'])

3 {'IoU': 0.0, 'CER': 1.0, 'Deteksi@0.5': 0.0, 'jumlah': 6}
5 {'IoU': 0.0, 'CER': 1.0, 'Deteksi@0.5': 0.0, 'jumlah': 6}
7 {'IoU': 0.0, 'CER': 1.0, 'Deteksi@0.5': 0.0, 'jumlah': 6}
9 {'IoU': 0.0, 'CER': 1.0, 'Deteksi@0.5': 0.0, 'jumlah': 6}


**Kesimpulan ablasi:** [isi]

## 3. Kasus gagal

Tiga citra dengan skor **IoU** terburuk dari metode terbaik. Analisis penyebabnya dan simpan gambarnya.

In [6]:
hasil_terbaik = ev.evaluasi_pelat(lambda c: metode_terbaik(c, NILAI_PARAMETER[0]), anotasi, FOLDER_CITRA, batas=UJI_CEPAT)
gagal = ev.terburuk(hasil_terbaik, 'IoU', n=3, makin_kecil_makin_baik=False)
FOLDER_FIGUR.mkdir(parents=True, exist_ok=True)
for r in gagal:
    print(r)
    citra = util.baca_citra(FOLDER_CITRA / r['berkas'])
    util.simpan_citra(FOLDER_FIGUR / f"gagal_{pathlib.Path(r['berkas']).stem}.png", util.ubah_ukuran_maks(citra, 1024))
util.tampil(*[util.baca_citra(FOLDER_CITRA / r['berkas']) for r in gagal], judul=[r['berkas'] for r in gagal])

{'berkas': 'uji_00.jpg', 'IoU': 0.0, 'CER': 1.0, 'teks': ''}
{'berkas': 'uji_01.jpg', 'IoU': 0.0, 'CER': 1.0, 'teks': ''}
{'berkas': 'uji_02.jpg', 'IoU': 0.0, 'CER': 1.0, 'teks': ''}


**Analisis kasus gagal:** [isi: untuk tiap citra, apa yang membuat metode gagal?]

## Simpan ke GitHub

1. **Runtime → Restart session and run all**, pastikan tidak ada error.
2. **File → Save a copy in GitHub** → repo `Prodi-TI-IBIKesatuan/pcd2026-uji-t1` → **branch Anda** (bukan `main`) → path tetap `notebooks/04_evaluasi.ipynb`.
3. Buka *pull request* ke `main` di github.com. Langkah lengkap: `PANDUAN.md`.